In [ ]:
import pandas as pd
import datetime
import chess
from utils import display_fen, get_db_connection
import matplotlib.pyplot as plt
import numpy as np
%load_ext autoreload
%autoreload 2

# create a personal database 
conn = get_db_connection(threads = 10)

# Define date range (midgame extraction will use these)
# games in 2019
start_date = "2023-12-01"
end_date = "2023-12-02"

#15 - 75 ply

In [ ]:
# Attach the primary database, ignore if already attached
db_path = '/scratch/gpfs/GRIFFITHS/chess-db/lichess.db'
try:
    conn.execute(f"ATTACH '{db_path}' AS core (READ_ONLY)")
except Exception as e:
    print(f"Warning attaching database: {e}")

# Create and load games of interest
conn.sql(f"""
    CREATE OR REPLACE TABLE midgame_positions AS
    SELECT m.*, g.*
    FROM core.moves m
    JOIN core.games g ON m.gid = g.gid
    WHERE g.utc_datetime BETWEEN '{start_date}' AND '{end_date}'
      AND g.initial_clock >= 300
      AND g.white_elo >= 2000
      AND g.black_elo >= 2000
      AND m.move_ply BETWEEN 20 AND 40
""")

# Note: The following is likely a bug in original code; changed to midgame_positions
filtered_positions = conn.sql("SELECT * FROM midgame_positions").df()

: 

In [ ]:
from sklearn.model_selection import train_test_split

# sample and split
sampled = filtered_positions[filtered_positions.game_end_type == "Normal"].sample(n=10000, random_state=42)
fit_positions, eval_positions = train_test_split(sampled, test_size=0.5, random_state=42)

In [ ]:
from chess.engine import EngineTerminatedError
from tqdm import tqdm
from utils import get_stockfish_engine
import numpy as np

SHALLOW_DEPTH = 1
DEEP_DEPTH = 15
TIME_LIMIT = 10.0

engine = get_stockfish_engine()
data = []

for i in tqdm(range(200)):
    position = eval_positions.iloc[i]

    board = chess.Board(position.board_position)
    try:
        player = 1 if board.turn == chess.WHITE else -1
        candidate_moves = set()
        shallow_move = None

        # clear the hash here so we get a fresh analysis each time
        engine.configure({"Clear Hash": True})
        with engine.analysis(board, chess.engine.Limit(depth=DEEP_DEPTH, time=TIME_LIMIT)) as analysis:
            for info in analysis:
                depth = info.get("depth")
                pv = info.get("pv")
                if pv:
                    if depth == SHALLOW_DEPTH:
                        shallow_move = pv[0]
                    candidate_moves.add(pv[0])

        if shallow_move is None or len(candidate_moves) == 0:
            continue
    
        engine.configure({"Clear Hash": True})
        infos = engine.analyse(
            board,
            chess.engine.Limit(depth=DEEP_DEPTH, time=TIME_LIMIT),
            multipv=len(candidate_moves),
            root_moves=list(candidate_moves)
        )

        scores = {}
        for info in infos:
            wdl = info["score"].white().wdl() if player == 1 else info["score"].black().wdl()
            scores[info["pv"][0]] = wdl.wins / 1000

        if shallow_move not in scores:
            continue

        v_shallow = scores[shallow_move]
        v_deep = max(scores.values())
        voc = v_deep - v_shallow

        data.append(pd.Series({
            "board_position": position.board_position,
            "player_white": position.player_white,
            "score_shallow": v_shallow,
            "score_deep": v_deep,
            "voc": voc,
            "move_time": position.move_time,
            "elo": position.white_elo if position.player_white else position.black_elo
        }))

    except EngineTerminatedError:
        print("Engine crashed on position", position.board_position)
        try:
            engine.quit()
        except Exception:
            pass
        engine = get_stockfish_engine()

engine.quit()

In [ ]:
df = pd.DataFrame(data)
df.voc.hist()

In [ ]:
# Equal-width bins — this is what Russek does
df["voc_bin"] = pd.cut(df["voc"], bins=7)  # adjust bin count
binned = df.groupby("voc_bin")["move_time"].mean()

plt.plot(range(len(binned)), binned.values, marker="o")
plt.xlabel("VOC (equal-width bins, low → high)")
plt.ylabel("Mean move time (s)")

In [ ]:
import statsmodels.formula.api as smf
m_linear = smf.ols("move_time ~ voc * elo", data=df).fit()
df["voc_sqrt"] = np.sqrt(df["voc"])
m_sqrt   = smf.ols("move_time ~ voc_sqrt * elo", data=df).fit()

In [ ]:
m_sqrt.summary()


In [ ]:
# how much better can we do? better model of VOC?
# architecture includes in it multiple steps - network has a decision to keep thinking or not
# if that model already explains more variance in RT than Evan's model
# each iteration is one-step lookahead vs each iteration is a full rollout

In [ ]:
# bin VOC and plot mean move time per bin
df["voc_bin"] = pd.qcut(df["voc"], q=10, duplicates="drop")
binned = df.groupby("voc_bin")["move_time"].mean()

plt.figure(figsize=(8, 4))
plt.plot(range(len(binned)), binned.values, marker="o")
plt.xlabel("VOC decile (low → high)")
plt.ylabel("Mean move time (s)")
plt.title("Mean move time by VOC bin")
plt.show()

In [ ]:
df = pd.DataFrame(data)

import matplotlib.pyplot as plt
plt.scatter(np.log(df.voc + 1e-5), np.log(df.move_time + 1e-5), alpha = 0.3)
correlation = df["voc"].corr(df["move_time"])
print("Correlation between VOC and move_time:", correlation)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(games.n_moves, bins = 30)
ax.set(title = "Number of moves in games", xlabel = "Number of moves", ylabel = "Number of games")

fig, ax = plt.subplots(figsize=(4, 5))
games.opening.value_counts().sort_values()[lambda x: x > 80].plot(kind="barh", ax=ax)